# Módulo 07 · Aula 03 — Testes

> **Manual de Estudos Interativo** · Trilha Engenharia de Software & Dados
> Projeto transversal: **Atlas / Aurora Comércio**

## A dor da Aurora

> *"Precisamos mudar o cálculo de frete. Mas ninguém quer mexer — a última vez que alteramos aquele arquivo, o relatório de faturamento quebrou e a gente só descobriu três dias depois."*

E aí vem a frase que define uma equipe sem testes:

> *"Está funcionando. **Não mexe.**"*

> 🎯 **Testes não existem para provar que o código funciona hoje.**
>
> Você já sabe que funciona — acabou de rodar e ver. Testes existem para você poder **mudar** o código amanhã e saber, em 4 segundos, se quebrou alguma coisa.
>
> A palavra certa não é *qualidade*. É **coragem**.

## O que você vai aprender aqui

| # | Tópico | Por que importa |
|---|--------|-----------------|
| 1 | `assert` e o primeiro teste | O básico, sem cerimônia |
| 2 | Fixtures | Preparo sem repetição |
| 3 | `conftest.py` | Onde as fixtures moram |
| 4 | **`dependency_overrides`** | 🎯 Testar com banco de mentira |
| 5 | Isolamento | 🔴 Testes que dependem da ordem |
| 6 | `parametrize` | 20 casos, uma função |
| 7 | `pytest.raises` | Testar o que dá errado |
| 8 | **`respx`** | API externa sem internet |
| 9 | O que **não** se testa | Onde o esforço se perde |
| 10 | Cobertura | 🔴 E a mentira que ela conta |

## ⚙️ Como este notebook funciona

Testes moram em **arquivos**, não em células. Vamos escrever arquivos de verdade com `%%writefile` e rodar o `pytest` de verdade sobre eles.

> 💭 **Por que não usar só `assert` nas células?** Porque o valor do pytest está no que ele faz **em volta** do `assert`: descoberta automática, fixtures, isolamento, relatório de falha legível e, principalmente, rodar tudo com um comando no CI (Módulo 09).

**Execute a célula abaixo antes de tudo.**

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  Preparação do Módulo 07 · Aula 03
# ═══════════════════════════════════════════════════════════════
import json
import shutil
import subprocess
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")


def _garantir(pacote, importar=None):
    nome = importar or pacote
    try:
        __import__(nome)
        return True
    except ImportError:
        print(f"  instalando {pacote}...")
        subprocess.run([sys.executable, "-m", "pip", "install", pacote, "--quiet"],
                       check=False, capture_output=True)
        try:
            __import__(nome)
            return True
        except ImportError:
            print(f"  ⚠️ {pacote} indisponível")
            return False


for _p, _m in [("fastapi", "fastapi"), ("httpx", "httpx"), ("pytest", "pytest"),
               ("sqlalchemy", "sqlalchemy"), ("respx", "respx"),
               ("python-multipart", "multipart")]:
    _garantir(_p, _m)

import pytest

print(f"✅ pytest {pytest.__version__}")


# ═══════════════════════════════════════════════════════════════
#  Projeto da aula
# ═══════════════════════════════════════════════════════════════
BASE = Path("aula_07_03").resolve()
if BASE.exists():
    shutil.rmtree(BASE)
(BASE / "app").mkdir(parents=True)
(BASE / "tests").mkdir(parents=True)
sys.path = [str(BASE)] + [p for p in sys.path if p != str(BASE)]
print(f"📁 {BASE}")


# ═══════════════════════════════════════════════════════════════
#  Rodar o pytest de verdade
# ═══════════════════════════════════════════════════════════════

def pytest_(*args, resumo: bool = True, linhas: int = 40) -> int:
    """Executa o pytest num subprocesso e mostra a saída.

    💡 Subprocesso, e não `pytest.main()`, por dois motivos:
       1. estado limpo a cada execução (o pytest guarda cache em memória)
       2. é literalmente o comando que você vai digitar no terminal
    """
    comando = [sys.executable, "-m", "pytest", "-p", "no:cacheprovider", *args]
    processo = subprocess.run(comando, cwd=BASE, capture_output=True, text=True)
    saida = (processo.stdout + processo.stderr).splitlines()

    if resumo and len(saida) > linhas:
        for linha in saida[:linhas // 2]:
            print(linha)
        print(f"   … (+{len(saida) - linhas} linhas)")
        for linha in saida[-(linhas // 2):]:
            print(linha)
    else:
        for linha in saida:
            print(linha)

    print(f"\n[saída do processo: {processo.returncode}]")
    return processo.returncode


def arvore(raiz: Path, prefixo=""):
    itens = [i for i in sorted(raiz.iterdir(), key=lambda p: (p.is_file(), p.name))
             if i.name not in {"__pycache__", ".pytest_cache"}]
    for i, item in enumerate(itens):
        ultimo = i == len(itens) - 1
        print(f"{prefixo}{'└── ' if ultimo else '├── '}{item.name}")
        if item.is_dir():
            arvore(item, prefixo + ("    " if ultimo else "│   "))


print("✅ `pytest_()` e `arvore()` prontos")

## 1. O primeiro teste

Um teste em pytest é uma função que começa com `test_` e usa `assert`. Só isso.

In [ ]:
%%writefile aula_07_03/app/frete.py
"""Cálculo de frete da Aurora — o código que ninguém quer mexer."""

FAIXAS_CEP = {"13": 2, "01": 3, "04": 3, "88": 6}
PESO_MAXIMO_KG = 30.0
FRETE_GRATIS_ACIMA_DE = 500.0


class FreteInvalido(ValueError):
    """Erro de domínio — o mesmo padrão do M01."""


def calcular_frete(peso_kg: float, cep: str, valor_pedido: float = 0.0) -> dict:
    """Devolve valor e prazo do frete.

    Regras (as que a Aurora combinou):
      · base R$ 12,90 + R$ 2,35 por quilo
      · prazo pela faixa de CEP; 8 dias se a faixa é desconhecida
      · frete grátis acima de R$ 500 — mas o prazo continua valendo
    """
    if peso_kg <= 0:
        raise FreteInvalido("peso deve ser positivo")
    if peso_kg > PESO_MAXIMO_KG:
        raise FreteInvalido(f"peso máximo é {PESO_MAXIMO_KG} kg")

    limpo = cep.replace("-", "").replace(".", "").strip()
    if len(limpo) != 8 or not limpo.isdigit():
        raise FreteInvalido(f"CEP inválido: {cep!r}")

    prazo = FAIXAS_CEP.get(limpo[:2], 8)
    valor = 12.90 + peso_kg * 2.35
    gratis = valor_pedido >= FRETE_GRATIS_ACIMA_DE

    return {"valor": 0.0 if gratis else round(valor, 2),
            "prazo_dias": prazo,
            "gratis": gratis}

In [ ]:
%%writefile aula_07_03/tests/test_frete.py
"""Testes do cálculo de frete."""
from app.frete import calcular_frete


def test_calcula_valor_basico():
    # ── Arrange: prepare os dados ──
    peso, cep = 2.0, "13010-000"

    # ── Act: execute UMA coisa ──
    resultado = calcular_frete(peso, cep)

    # ── Assert: verifique ──
    assert resultado["valor"] == 17.60      # 12.90 + 2*2.35
    assert resultado["prazo_dias"] == 2
    assert resultado["gratis"] is False


def test_frete_gratis_acima_do_limite():
    resultado = calcular_frete(2.0, "13010-000", valor_pedido=600.0)
    assert resultado["valor"] == 0.0
    assert resultado["gratis"] is True
    # 🎯 O prazo NÃO muda por ser grátis — este assert protege a regra
    assert resultado["prazo_dias"] == 2


def test_cep_desconhecido_usa_prazo_padrao():
    assert calcular_frete(1.0, "99999-999")["prazo_dias"] == 8

In [ ]:
pytest_("-v")

> 💡 **O padrão AAA — Arrange, Act, Assert.**
>
> Prepare, execute **uma** coisa, verifique. Quando um teste tem dois "Act", ele testa duas coisas — e quando falha, você não sabe qual.
>
> 🎯 **O nome do teste é documentação.** `test_frete_gratis_acima_do_limite` diz o que a regra é. Compare com `test_frete_2`: quando ele falhar no CI às 3h da manhã, qual dos dois você prefere ler?
>
> ⚠️ **`assert resultado["gratis"] is False`, não `== False`.** O `==` aceitaria `0`, `""` ou `[]`. O `is` exige o booleano.

Uma falha bem apresentada vale mais do que dez comentários. Veja como o pytest mostra.

In [ ]:
%%writefile aula_07_03/tests/test_falha_proposital.py
from app.frete import calcular_frete


def test_que_falha_de_proposito():
    resultado = calcular_frete(2.0, "13010-000")
    assert resultado["valor"] == 99.99      # 🔴 errado de propósito

In [ ]:
pytest_("tests/test_falha_proposital.py", "-v", resumo=False)

> 🎯 **Repare no que o pytest mostrou:** ele reescreve o `assert` para exibir os **dois lados** da comparação e o dicionário inteiro.
>
> Isso é o `assert` reescrito (*assertion rewriting*) — a razão pela qual você **não** precisa de `self.assertEqual(a, b)` como em outros frameworks. `assert a == b` já dá a mensagem completa.

In [ ]:
import os
os.remove(BASE / "tests" / "test_falha_proposital.py")
print("✅ teste proposital removido")

## 2. Fixtures

Repetir preparo em todo teste é o caminho mais curto para ninguém mais escrever testes.

In [ ]:
%%writefile aula_07_03/tests/test_fixtures.py
"""Fixtures: preparo reaproveitável."""
import pytest

from app.frete import FreteInvalido, calcular_frete


# ═══ Uma fixture é uma função com @pytest.fixture ═══
@pytest.fixture
def pedido_padrao() -> dict:
    """Dados de um pedido típico da Aurora."""
    return {"peso_kg": 2.4, "cep": "13010-000", "valor_pedido": 250.0}


@pytest.fixture
def pedido_grande(pedido_padrao) -> dict:
    """🔑 Fixtures compõem: esta usa a anterior."""
    return {**pedido_padrao, "valor_pedido": 900.0}


def test_pedido_padrao_paga_frete(pedido_padrao):
    assert calcular_frete(**pedido_padrao)["gratis"] is False


def test_pedido_grande_nao_paga(pedido_grande):
    assert calcular_frete(**pedido_grande)["gratis"] is True


# ═══ Fixture com limpeza: yield ═══
@pytest.fixture
def arquivo_temporario(tmp_path):
    """`tmp_path` é uma fixture do próprio pytest: pasta nova por teste."""
    caminho = tmp_path / "dados.csv"
    caminho.write_text("sku,qtd\nNB-01,5\n", encoding="utf-8")
    yield caminho
    # ── tudo depois do yield roda DEPOIS do teste, mesmo se ele falhar ──
    if caminho.exists():
        caminho.unlink()


def test_le_arquivo(arquivo_temporario):
    conteudo = arquivo_temporario.read_text(encoding="utf-8")
    assert "NB-01" in conteudo


# ═══ 🔴 A armadilha do escopo ═══
@pytest.fixture(scope="module")
def lista_compartilhada() -> list:
    """scope="module": UMA instância para o arquivo inteiro.

    🔴 Objeto MUTÁVEL com escopo amplo é uma armadilha: um teste altera
       e o próximo recebe o estado sujo. A ordem passa a importar.
    """
    return []


def test_a_adiciona(lista_compartilhada):
    lista_compartilhada.append("a")
    assert len(lista_compartilhada) == 1


def test_b_ve_o_lixo_do_anterior(lista_compartilhada):
    # 🔴 Se este arquivo rodar sozinho, passa. Junto com o de cima,
    #    a lista já tem "a". Este assert DOCUMENTA o problema.
    assert lista_compartilhada == ["a"]

In [ ]:
pytest_("tests/test_fixtures.py", "-v", resumo=False)

> 🔴 **`test_b_ve_o_lixo_do_anterior` passou — e isso é péssimo.**
>
> Ele só passa porque `test_a_adiciona` rodou antes. Rode-o sozinho com `pytest -k test_b` e ele **falha**.
>
> Um teste que depende da ordem é pior do que nenhum teste: ele falha em situações que não têm nada a ver com o bug, e você aprende a ignorá-lo.
>
> 🧭 **A regra dos escopos:**
>
> | Escopo | Criado | Use para |
> |--------|--------|----------|
> | `function` (padrão) | a cada teste | ✅ quase tudo |
> | `class` | por classe | agrupamentos |
> | `module` | por arquivo | 🔶 recursos caros e **imutáveis** |
> | `session` | uma vez | 🔶 engine de banco, container Docker |
>
> **Escopo amplo só para o que é caro de criar E não muda.** Criar o `Engine` do SQLAlchemy? `session`. A **sessão** dele? `function`, sempre.

## 3. `conftest.py` — onde as fixtures moram

In [ ]:
%%writefile aula_07_03/app/main.py
"""A API do Atlas, versão enxuta para os testes."""
from typing import Annotated

from fastapi import Depends, FastAPI, HTTPException, Query, status
from pydantic import BaseModel, Field
from sqlalchemy import String, create_engine, select
from sqlalchemy.orm import DeclarativeBase, Mapped, Session, mapped_column, sessionmaker

from app.frete import FreteInvalido, calcular_frete


class Base(DeclarativeBase):
    pass


class Produto(Base):
    __tablename__ = "produtos"
    id: Mapped[int] = mapped_column(primary_key=True)
    sku: Mapped[str] = mapped_column(String(20), unique=True, index=True)
    nome: Mapped[str] = mapped_column(String(120))
    preco: Mapped[float]
    custo: Mapped[float]
    estoque: Mapped[int] = mapped_column(default=0)


# ── Esquemas ──
class ProdutoCriar(BaseModel):
    sku: str = Field(min_length=5, max_length=20, pattern=r"^[A-Z]{2}-[A-Z0-9-]+$")
    nome: str = Field(min_length=3, max_length=120)
    preco: float = Field(gt=0)
    custo: float = Field(ge=0)
    estoque: int = Field(default=0, ge=0)


class ProdutoResposta(BaseModel):
    model_config = {"from_attributes": True}
    sku: str
    nome: str
    preco: float
    estoque: int
    # 🔒 sem `custo`


class Cotacao(BaseModel):
    peso_kg: float = Field(gt=0)
    cep: str
    valor_pedido: float = Field(default=0.0, ge=0)


# ── Banco (o padrão do M06) ──
motor = create_engine("sqlite:///./atlas_dev.db")
Sessao = sessionmaker(bind=motor, expire_on_commit=False)


def obter_sessao():
    sessao = Sessao()
    try:
        yield sessao
    finally:
        sessao.close()


SessaoDep = Annotated[Session, Depends(obter_sessao)]


def criar_app() -> FastAPI:
    app = FastAPI(title="Atlas")

    @app.get("/saude")
    def saude():
        return {"status": "ok"}

    @app.post("/produtos", response_model=ProdutoResposta, status_code=201)
    def criar(dados: ProdutoCriar, sessao: SessaoDep):
        if sessao.scalar(select(Produto).where(Produto.sku == dados.sku)):
            raise HTTPException(status.HTTP_409_CONFLICT, "SKU já existe")
        if dados.preco < dados.custo:
            raise HTTPException(422, "preço abaixo do custo")
        produto = Produto(**dados.model_dump())
        sessao.add(produto)
        sessao.commit()
        return produto

    @app.get("/produtos", response_model=list[ProdutoResposta])
    def listar(sessao: SessaoDep, limite: Annotated[int, Query(ge=1, le=100)] = 20):
        return list(sessao.scalars(select(Produto).order_by(Produto.sku).limit(limite)))

    @app.get("/produtos/{sku}", response_model=ProdutoResposta)
    def obter(sku: str, sessao: SessaoDep):
        produto = sessao.scalar(select(Produto).where(Produto.sku == sku))
        if produto is None:
            raise HTTPException(404, "não encontrado")
        return produto

    @app.post("/cotacoes")
    def cotar(dados: Cotacao):
        try:
            return calcular_frete(dados.peso_kg, dados.cep, dados.valor_pedido)
        except FreteInvalido as erro:
            raise HTTPException(422, str(erro)) from erro

    return app

In [ ]:
%%writefile aula_07_03/tests/conftest.py
"""Fixtures compartilhadas por TODOS os testes desta pasta.

🔑 O pytest encontra este arquivo sozinho — nada de importar.
   Ele também é o lugar certo para ajustar o `sys.path`.
"""
import pytest
from fastapi.testclient import TestClient
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker
from sqlalchemy.pool import StaticPool

from app.main import Base, Produto, criar_app, obter_sessao


@pytest.fixture
def sessao():
    """🎯 Um banco NOVO, em memória, para cada teste.

    Isolamento perfeito pela via mais simples que existe: o banco não é
    limpo, ele é *recriado*. Não há estado para vazar entre testes,
    independentemente da ordem em que eles rodem.

    🔴 `poolclass=StaticPool` é obrigatório com SQLite em memória.
       Um banco `:memory:` pertence à CONEXÃO. Sem o StaticPool, o
       TestClient — que roda as rotas noutra thread — abre uma conexão
       NOVA e encontra um banco vazio. O erro é
       `no such table: produtos`, que parece problema de modelo e não é.

    💭 "Mas criar um engine por teste não é caro?" Para SQLite em
       memória, custa cerca de 1 ms. Para PostgreSQL seria caro — e é
       por isso que lá se usa outro padrão, mostrado logo abaixo.
    """
    motor = create_engine(
        "sqlite:///:memory:",
        connect_args={"check_same_thread": False},
        poolclass=StaticPool,
    )
    Base.metadata.create_all(motor)
    Sessao = sessionmaker(bind=motor, expire_on_commit=False)
    s = Sessao()
    yield s
    s.close()
    motor.dispose()


@pytest.fixture
def cliente(sessao):
    """TestClient com o banco de teste no lugar do real.

    🎯 Isto só é possível porque a rota DECLARA que precisa de uma
       sessão (`Depends`) em vez de criar uma. É o retorno concreto do
       trabalho de arquitetura do M06.
    """
    app = criar_app()
    app.dependency_overrides[obter_sessao] = lambda: sessao
    with TestClient(app) as c:
        yield c
    app.dependency_overrides.clear()


@pytest.fixture
def catalogo(sessao):
    """Três produtos já no banco."""
    produtos = [
        Produto(sku="NB-DELL-15", nome="Notebook Dell", preco=2599.90,
                custo=2120.0, estoque=14),
        Produto(sku="MO-LG-24UW", nome="Monitor LG", preco=1199.0,
                custo=920.0, estoque=31),
        Produto(sku="PE-LOG-M170", nome="Mouse Logitech", preco=89.90,
                custo=52.0, estoque=0),
    ]
    sessao.add_all(produtos)
    sessao.commit()
    return produtos

In [ ]:
%%writefile aula_07_03/tests/test_api.py
"""Testes da API — usando as fixtures do conftest."""
import pytest


def test_saude(cliente):
    resposta = cliente.get("/saude")
    assert resposta.status_code == 200
    assert resposta.json() == {"status": "ok"}


def test_lista_vazia_no_inicio(cliente):
    """🎯 Este teste PROVA o isolamento.

    Ele só passa se o banco estiver limpo — mesmo que outro teste tenha
    criado produtos antes. Se ele começar a falhar, o seu isolamento
    quebrou.
    """
    assert cliente.get("/produtos").json() == []


def test_cria_produto(cliente):
    resposta = cliente.post("/produtos", json={
        "sku": "NB-DELL-15", "nome": "Notebook Dell",
        "preco": 2599.90, "custo": 2120.0, "estoque": 14})
    assert resposta.status_code == 201
    assert resposta.json()["sku"] == "NB-DELL-15"


def test_resposta_nao_expoe_custo(cliente, catalogo):
    """🔒 O teste de segurança mais barato que existe."""
    corpo = cliente.get("/produtos/NB-DELL-15").json()
    assert "custo" not in corpo
    assert "custo" not in cliente.get("/produtos").json()[0]


def test_sku_duplicado_da_409(cliente, catalogo):
    resposta = cliente.post("/produtos", json={
        "sku": "NB-DELL-15", "nome": "Duplicado",
        "preco": 100.0, "custo": 50.0})
    assert resposta.status_code == 409


def test_produto_inexistente_da_404(cliente):
    assert cliente.get("/produtos/ZZ-NAO-EXISTE").status_code == 404


def test_catalogo_esta_no_banco(cliente, catalogo):
    corpo = cliente.get("/produtos").json()
    assert len(corpo) == 3
    assert {p["sku"] for p in corpo} == {"NB-DELL-15", "MO-LG-24UW", "PE-LOG-M170"}

In [ ]:
pytest_("tests/test_api.py", "-v", resumo=False)

> 🎯 **O `test_lista_vazia_no_inicio` é o teste mais importante deste arquivo** — e ele não testa nenhuma regra de negócio.
>
> Ele testa a sua **infraestrutura de testes**. Enquanto ele passar, você sabe que cada teste começa do zero. No dia em que ele falhar, o problema não é o código da aplicação: é que os seus testes começaram a se contaminar.
>
> 💡 **Repare também na ordem:** `test_catalogo_esta_no_banco` cria 3 produtos, e `test_lista_vazia_no_inicio` continua vendo o banco vazio — rodando junto ou sozinho, em qualquer ordem. É isso que a fixture garante.

In [ ]:
# Prova: selecionando só dois testes, em qualquer ordem, ambos passam
pytest_("tests/test_api.py", "-v", "--tb=short",
        "-k", "catalogo_esta or lista_vazia", resumo=False)

### 3.1 — O outro padrão de isolamento (e a armadilha do SQLite)

Recriar o banco a cada teste é perfeito para SQLite em memória. Para **PostgreSQL**, criar e destruir o schema a cada teste custaria segundos — inviável com 500 testes.

O padrão de lá é outro: **uma transação por teste, revertida no fim**.

```
conexao = motor.connect()
transacao = conexao.begin()          ← abre a transação EXTERNA
    sessão amarrada a essa conexão
    a aplicação faz commit()          ← vira um SAVEPOINT, não um commit real
transacao.rollback()                 ← 🔑 descarta tudo
```

O banco nunca chega a gravar nada. É rápido e funciona com qualquer volume de dados de teste.

In [ ]:
%%writefile aula_07_03/tests/test_isolamento_por_transacao.py
"""O padrão de PostgreSQL — e o que ele exige no SQLite.

🔴 A ARMADILHA QUE ME PEGOU AO ESCREVER ESTA AULA

   Este padrão, copiado da documentação e aplicado direto no SQLite,
   NÃO FUNCIONA. O `rollback()` não desfaz nada, e o segundo teste que
   usa a mesma fixture estoura com

       IntegrityError: UNIQUE constraint failed

   O motivo não é o SQLAlchemy: é o driver `pysqlite`, da biblioteca
   padrão do Python. Ele gerencia transações por conta própria, emite
   BEGIN nas horas erradas e faz commits implícitos — e isso quebra os
   SAVEPOINTs em que o padrão se apoia.

   A correção são os dois `event.listens_for` abaixo: desligamos o
   controle de transação do pysqlite e emitimos o BEGIN nós mesmos.

💭 Em PostgreSQL nada disso é necessário — o `psycopg` se comporta.
   Guarde este arquivo para quando você migrar os testes para lá.
"""
import pytest
from sqlalchemy import String, create_engine, event, select
from sqlalchemy.orm import sessionmaker
from sqlalchemy.pool import StaticPool

from app.main import Base, Produto


@pytest.fixture(scope="module")
def motor_persistente():
    """Criado UMA vez: é o que torna o padrão vantajoso."""
    motor = create_engine("sqlite:///:memory:",
                          connect_args={"check_same_thread": False},
                          poolclass=StaticPool)

    # 🔴 Sem estes dois ganchos, o rollback abaixo não desfaz nada.
    @event.listens_for(motor, "connect")
    def _desligar_transacao_implicita(dbapi, registro):
        dbapi.isolation_level = None

    @event.listens_for(motor, "begin")
    def _begin_explicito(conexao):
        conexao.exec_driver_sql("BEGIN")

    Base.metadata.create_all(motor)
    yield motor
    motor.dispose()


@pytest.fixture
def sessao_transacional(motor_persistente):
    conexao = motor_persistente.connect()
    transacao = conexao.begin()
    Sessao = sessionmaker(bind=conexao, expire_on_commit=False,
                          join_transaction_mode="create_savepoint")
    s = Sessao()
    yield s
    s.close()
    transacao.rollback()          # 🔑 descarta tudo que o teste fez
    conexao.close()


def _semear(sessao):
    sessao.add(Produto(sku="TR-UNICO-1", nome="Teste", preco=10.0,
                       custo=5.0, estoque=1))
    sessao.commit()               # ← commit da APLICAÇÃO, vira savepoint


def test_primeiro_grava(sessao_transacional):
    _semear(sessao_transacional)
    assert sessao_transacional.scalars(select(Produto)).all()


def test_segundo_grava_o_mesmo_sku(sessao_transacional):
    """🎯 Só passa se o rollback do teste anterior funcionou.

    Se ele estourar com UNIQUE constraint, o isolamento está quebrado.
    """
    _semear(sessao_transacional)
    assert len(sessao_transacional.scalars(select(Produto)).all()) == 1


def test_terceiro_ve_banco_vazio(sessao_transacional):
    assert sessao_transacional.scalars(select(Produto)).all() == []

In [ ]:
pytest_("tests/test_isolamento_por_transacao.py", "-v", resumo=False)

> 🔴 **Esta armadilha é real e me custou uma sessão de depuração.**
>
> O padrão de rollback é o mais recomendado da internet — e ele falha silenciosamente no SQLite. Silenciosamente porque o *primeiro* teste passa: você só descobre quando o segundo usa o mesmo dado.
>
> 🧭 **A escolha prática:**
>
> | Banco de teste | Padrão |
> |----------------|--------|
> | SQLite em memória | ✅ recriar o banco por teste (simples, imbatível) |
> | PostgreSQL | ✅ transação + rollback (rápido em escala) |
> | SQLite + rollback | ⚠️ funciona, mas exige os ganchos do `pysqlite` |
>
> 💭 **A lição maior:** quando um padrão "de livro" não funciona, o problema costuma estar uma camada abaixo do que você está olhando. Aqui não era o SQLAlchemy — era o driver.

## 4. `parametrize` — muitos casos, uma função

In [ ]:
%%writefile aula_07_03/tests/test_parametrizado.py
"""Um caso por linha, em vez de uma função por caso."""
import pytest

from app.frete import FreteInvalido, calcular_frete


@pytest.mark.parametrize("peso,cep,valor_esperado", [
    (1.0,  "13010-000", 15.25),
    (2.0,  "13010-000", 17.60),
    (10.0, "13010-000", 36.40),
    (0.5,  "01310-100", 14.08),
    (30.0, "88010-000", 83.40),      # peso máximo permitido
])
def test_valores_do_frete(peso, cep, valor_esperado):
    assert calcular_frete(peso, cep)["valor"] == pytest.approx(valor_esperado)


@pytest.mark.parametrize("cep,prazo", [
    ("13010-000", 2),      # Campinas
    ("01310-100", 3),      # São Paulo
    ("04567-000", 3),
    ("88010-000", 6),      # Florianópolis
    ("99999-999", 8),      # faixa desconhecida → padrão
])
def test_prazos_por_faixa(cep, prazo):
    assert calcular_frete(1.0, cep)["prazo_dias"] == prazo


# ═══ 🔴 Os casos de ERRO merecem tanto cuidado quanto os de sucesso ═══
@pytest.mark.parametrize("peso,cep,pedaco_da_mensagem", [
    (0,    "13010-000", "positivo"),
    (-5,   "13010-000", "positivo"),
    (30.1, "13010-000", "peso máximo"),
    (1.0,  "1301000",   "CEP inválido"),      # 7 dígitos
    (1.0,  "13010-0000", "CEP inválido"),     # 9 dígitos
    (1.0,  "abcde-fgh",  "CEP inválido"),
    (1.0,  "",           "CEP inválido"),
])
def test_entradas_invalidas(peso, cep, pedaco_da_mensagem):
    with pytest.raises(FreteInvalido, match=pedaco_da_mensagem):
        calcular_frete(peso, cep)


# ═══ IDs legíveis: o nome do teste vira documentação ═══
@pytest.mark.parametrize("valor_pedido,gratis", [
    pytest.param(499.99, False, id="abaixo_do_limite"),
    pytest.param(500.00, True,  id="exatamente_no_limite"),
    pytest.param(500.01, True,  id="acima_do_limite"),
])
def test_limite_do_frete_gratis(valor_pedido, gratis):
    """🎯 As BORDAS são onde os bugs moram.

    `>=` ou `>`? Um teste em cada lado do limite responde para sempre.
    """
    assert calcular_frete(1.0, "13010-000", valor_pedido)["gratis"] is gratis

In [ ]:
pytest_("tests/test_parametrizado.py", "-v", resumo=False, linhas=60)

> 🎯 **`pytest.approx` para números decimais.** `12.90 + 2*2.35 == 17.6` é `False` em ponto flutuante. O `approx` compara com tolerância — e é por isso que dinheiro em banco é `NUMERIC`, não `FLOAT` (M05).
>
> 🎯 **`pytest.raises(..., match=...)`** verifica o tipo **e** a mensagem. Sem o `match`, um `FreteInvalido` levantado pelo motivo errado passaria no teste.
>
> 💡 **`pytest.param(..., id="...")`** faz o relatório dizer `test_limite_do_frete_gratis[exatamente_no_limite]`. Quando falhar, você já sabe qual caso quebrou sem abrir o arquivo.

## 5. `respx` — testar sem a internet

O código que chama a transportadora (aula 07_01) também precisa de teste. Mas um teste não pode depender de a API alheia estar no ar.

In [ ]:
%%writefile aula_07_03/app/transportadora.py
"""Cliente da Transportadora Veloz (versão da aula 07_01, resumida)."""
import httpx

URL_BASE = "https://api.veloz.com.br"


class TransportadoraIndisponivel(Exception):
    """Erro de domínio — a API não sabe o que é `httpx`."""


def cotar_frete(peso_kg: float, cep: str, tentativas: int = 3) -> dict:
    """Cota o frete, com retry para falhas transitórias."""
    ultima = None
    with httpx.Client(base_url=URL_BASE, timeout=5.0) as cliente:
        for _ in range(tentativas):
            try:
                r = cliente.post("/v1/cotacoes",
                                 json={"peso_kg": peso_kg, "cep_destino": cep})
            except httpx.RequestError as erro:
                ultima = erro
                continue
            if r.status_code in (500, 502, 503, 504):
                ultima = httpx.HTTPStatusError(
                    f"status {r.status_code}", request=r.request, response=r)
                continue
            if r.status_code == 422:
                raise ValueError(r.json().get("detail", "dados inválidos"))
            r.raise_for_status()
            return r.json()
    raise TransportadoraIndisponivel(str(ultima))

In [ ]:
%%writefile aula_07_03/tests/test_transportadora.py
"""Testes do cliente HTTP — sem tocar na internet.

🔑 O `respx` intercepta o httpx no nível do TRANSPORTE. O seu código
   roda inteiro, sem alteração nenhuma: monta a requisição, serializa
   o JSON, aplica o timeout, trata o status. Só o socket não existe.

💭 Compare com `unittest.mock.patch("httpx.post")`: aquilo substitui a
   FUNÇÃO, e aí você deixa de testar tudo que a função faria. Se o seu
   código passar `params` errados, o mock não percebe — o respx sim.
"""
import json

import httpx
import pytest
import respx

from app.transportadora import TransportadoraIndisponivel, cotar_frete

URL = "https://api.veloz.com.br/v1/cotacoes"


@respx.mock
def test_cotacao_bem_sucedida():
    rota = respx.post(URL).mock(
        return_value=httpx.Response(200, json={"valor": 18.54, "prazo_dias": 2}))

    resultado = cotar_frete(2.4, "13010-000")

    assert resultado["valor"] == 18.54
    # 🎯 Verifique o que FOI ENVIADO, não só o que voltou
    assert rota.called
    assert rota.call_count == 1
    enviado = json.loads(rota.calls[0].request.content)
    assert enviado == {"peso_kg": 2.4, "cep_destino": "13010-000"}


@respx.mock
def test_repete_em_erro_transitorio():
    """🎯 Respostas em SEQUÊNCIA: falha, falha, sucesso."""
    respx.post(URL).mock(side_effect=[
        httpx.Response(503),
        httpx.Response(503),
        httpx.Response(200, json={"valor": 18.54, "prazo_dias": 2}),
    ])

    assert cotar_frete(2.4, "13010-000")["valor"] == 18.54


@respx.mock
def test_desiste_apos_as_tentativas():
    rota = respx.post(URL).mock(return_value=httpx.Response(503))

    with pytest.raises(TransportadoraIndisponivel):
        cotar_frete(2.4, "13010-000")

    assert rota.call_count == 3          # 🔑 tentou exatamente 3 vezes


@respx.mock
def test_erro_de_rede_tambem_e_repetido():
    rota = respx.post(URL).mock(side_effect=httpx.ConnectError("recusada"))

    with pytest.raises(TransportadoraIndisponivel):
        cotar_frete(2.4, "13010-000")

    assert rota.call_count == 3


@respx.mock
def test_nao_repete_erro_de_validacao():
    """🔴 422 é culpa NOSSA. Repetir é teimosia."""
    rota = respx.post(URL).mock(
        return_value=httpx.Response(422, json={"detail": "cep_destino inválido"}))

    with pytest.raises(ValueError, match="cep_destino"):
        cotar_frete(1.0, "SEM-CEP")

    assert rota.call_count == 1          # 🎯 UMA vez, sem retry


@respx.mock
def test_timeout_conta_como_indisponivel():
    respx.post(URL).mock(side_effect=httpx.ReadTimeout("demorou"))
    with pytest.raises(TransportadoraIndisponivel):
        cotar_frete(1.0, "13010-000")

In [ ]:
pytest_("tests/test_transportadora.py", "-v", resumo=False)

> 🎯 **`assert rota.call_count == 1` no teste do 422 é o melhor teste do arquivo.**
>
> Ele não verifica o resultado — verifica **o que o seu código não fez**. Se alguém amanhã adicionar `422` à lista de status repetíveis, este teste quebra e explica o porquê.
>
> Testar comportamento ausente é raro e valioso.
>
> ⚠️ **Cuidado com o excesso de mock.** Se você simular tanto que só sobra a sua própria lógica de controle, o teste vira uma tautologia: ele confirma que o código faz o que o código faz. O respx acerta o nível — simula a **rede**, não o seu cliente.

## 6. Marcadores e organização

In [ ]:
%%writefile aula_07_03/pyproject.toml
[tool.pytest.ini_options]
testpaths = ["tests"]
python_files = ["test_*.py"]
addopts = "-q --strict-markers"

# 🔑 --strict-markers: um marcador com erro de digitação vira ERRO,
#    não um teste silenciosamente ignorado. Sem isso, `@pytest.mark.lentoo`
#    simplesmente não faz nada e você nunca descobre.
markers = [
    "lento: demora mais de 1 segundo",
    "integracao: precisa de banco, rede ou serviço externo",
    "seguranca: verifica uma propriedade de segurança",
]

In [ ]:
%%writefile aula_07_03/tests/test_marcadores.py
"""Nem todo teste roda sempre."""
import sys
import time

import pytest


@pytest.mark.seguranca
def test_custo_nunca_sai_na_resposta(cliente, catalogo):
    for corpo in [cliente.get("/produtos/NB-DELL-15").json(),
                  *cliente.get("/produtos").json()]:
        assert "custo" not in corpo


@pytest.mark.seguranca
def test_query_invalida_e_recusada(cliente):
    assert cliente.get("/produtos?limite=99999").status_code == 422
    assert cliente.get("/produtos?limite=0").status_code == 422


@pytest.mark.lento
def test_que_demora():
    time.sleep(1.2)
    assert True


@pytest.mark.integracao
@pytest.mark.skip(reason="exige a API real da transportadora")
def test_contra_a_api_de_verdade():
    """💭 Testes de contrato existem — mas não rodam a cada commit.

    Eles rodam uma vez por dia, num job separado, e servem para
    descobrir que o parceiro mudou a API antes que o cliente descubra.
    """
    ...


@pytest.mark.skipif(sys.platform == "win32", reason="caminhos POSIX")
def test_so_em_unix():
    assert "/" in "/tmp/x"


@pytest.mark.xfail(reason="bug conhecido #142, correção agendada")
def test_bug_conhecido():
    """`xfail` documenta um bug SEM deixar o CI vermelho.

    ⚠️ E, se ele passar inesperadamente, o pytest avisa (XPASS) — é
       assim que você descobre que o bug foi corrigido.
    """
    assert 1 == 2

In [ ]:
print("═══ Tudo, menos o que é lento ═══")
pytest_("-m", "not lento", "-v", resumo=False, linhas=60)

In [ ]:
print("═══ Só os testes de segurança ═══")
pytest_("-m", "seguranca", "-v", resumo=False)

> 💡 **Marcadores servem para dividir o tempo.**
>
> | Quando | O que roda |
> |--------|-----------|
> | A cada salvamento | `-m "not lento and not integracao"` — 2 segundos |
> | Antes do commit | tudo, menos integração |
> | No CI | tudo |
> | Uma vez por dia | testes de contrato contra as APIs reais |
>
> 🔴 **`--strict-markers` não é opcional.** Sem ele, `@pytest.mark.lentoo` (com erro de digitação) não faz absolutamente nada — e o seu teste lento continua rodando em toda salvada, ou pior, um teste que você achou que estava sendo pulado nunca foi.

## 7. Cobertura — e a mentira que ela conta

In [ ]:
if _garantir("pytest-cov", "pytest_cov"):
    pytest_("--cov=app", "--cov-report=term-missing",
            "-m", "not lento", resumo=False, linhas=50)
else:
    print("⚠️ pytest-cov indisponível neste ambiente")

Agora o contraexemplo: **100% de cobertura com zero valor**.

In [ ]:
%%writefile aula_07_03/tests/test_cobertura_enganosa.py
"""Este arquivo cobre 100% de `frete.py` e não testa NADA."""
from app.frete import calcular_frete


def test_cobre_tudo_sem_verificar_nada():
    # Passa por todas as linhas...
    calcular_frete(2.0, "13010-000")
    calcular_frete(2.0, "13010-000", valor_pedido=600)
    calcular_frete(1.0, "99999-999")
    for peso, cep in [(0, "13010-000"), (99, "13010-000"), (1, "x")]:
        try:
            calcular_frete(peso, cep)
        except Exception:
            pass
    # ...e não afirma coisa nenhuma. 🔴
    assert True

In [ ]:
pytest_("tests/test_cobertura_enganosa.py", "--cov=app.frete",
        "--cov-report=term-missing", resumo=False)

> 🔴 **Cobertura mede linhas EXECUTADAS, não comportamento VERIFICADO.**
>
> O arquivo acima executa quase todo o `frete.py` e não afirma nada. Troque `12.90` por `99.90` no código: a cobertura continua igual, e o teste continua passando.
>
> 🧭 **Como ler a cobertura corretamente:**
>
> | Leitura | Vale? |
> |---------|-------|
> | "Cobertura baixa → há código não testado" | ✅ **verdade útil** |
> | "Cobertura alta → o código está testado" | 🔴 **não se conclui** |
>
> Ela é um **detector de buracos**, não um certificado de qualidade.
>
> ⚠️ **E a meta de 80% no CI?** Ela é útil como piso — impede que alguém adicione um módulo inteiro sem nenhum teste. Mas quando vira objetivo, as pessoas escrevem testes como o de cima. *"Quando uma métrica vira meta, ela deixa de ser uma boa métrica."* (Lei de Goodhart.)

In [ ]:
import os

os.remove(BASE / "tests" / "test_cobertura_enganosa.py")
(BASE / ".coverage").unlink(missing_ok=True)      # 💡 vai no .gitignore
print("✅ removido")

## 8. O que testar — e o que não

In [ ]:
avaliacao = [
    ("Regra de negócio (cálculo de frete, curva ABC)",   "SIM ", "é o coração"),
    ("Bordas e limites (>=  vs >, lista vazia, zero)",   "SIM ", "onde os bugs moram"),
    ("Caminhos de erro (404, 409, 422)",                 "SIM ", "quase sempre esquecidos"),
    ("Contratos de saída (custo não vaza)",              "SIM ", "🔒 segurança barata"),
    ("Autorização (leitor não apaga)",                   "SIM ", "🔒 regressão cara"),
    ("Atomicidade (pedido inviável não baixa estoque)",  "SIM ", "🔴 dinheiro"),
    ("Integração com API externa (com respx)",           "SIM ", "sem depender da rede"),
    ("Bug corrigido (teste que falha antes da correção)", "SIM ", "impede o retorno"),
    ("", "", ""),
    ("Getters e setters triviais",                       "nao ", "testa a linguagem"),
    ("Que o Pydantic valida `gt=0`",                     "nao ", "testa a biblioteca"),
    ("Que o SQLAlchemy grava no banco",                  "nao ", "testa a biblioteca"),
    ("Detalhes internos privados",                       "nao ", "trava a refatoração"),
    ("Que o FastAPI devolve 422",                        "nao ", "testa o framework"),
    ("O texto exato de uma mensagem de log",             "nao ", "quebra à toa"),
]
print(f"{'O quê':<52}{'Testar?':<9}Por quê")
print("─" * 88)
for item, veredito, motivo in avaliacao:
    if not item:
        print()
        continue
    print(f"{item:<52}{veredito:<9}{motivo}")

> 🎯 **Teste comportamento, não implementação.**
>
> Um teste que verifica *"a função `_calcular_base` foi chamada com 2.35"* quebra quando você renomeia o método — mesmo sem nenhum bug. Um teste que verifica *"frete de 2 kg para Campinas custa R$ 17,60"* continua valendo depois de qualquer refatoração.
>
> **A régua:** se refatorar sem mudar comportamento quebra o teste, o teste está testando a coisa errada.
>
> 💭 **E o teste de regressão?** Quando um bug aparece em produção, o primeiro passo não é corrigir — é **escrever um teste que falha por causa dele**. Aí você corrige e vê o teste passar. Sem isso, você não tem certeza de que corrigiu, nem garantia de que não volta.

## 🔧 Prática guiada — a suíte do Atlas

In [ ]:
%%writefile aula_07_03/tests/test_atlas_completo.py
"""A suíte que dá coragem para mexer no Atlas."""
import httpx
import pytest
import respx


# ═══════════════════════════════════════════════════════════════
#  Contrato público
# ═══════════════════════════════════════════════════════════════
def test_saude_responde(cliente):
    assert cliente.get("/saude").status_code == 200


@pytest.mark.parametrize("caminho", ["/produtos", "/produtos/NB-DELL-15", "/saude"])
def test_rotas_de_leitura_existem(cliente, catalogo, caminho):
    assert cliente.get(caminho).status_code == 200


# ═══════════════════════════════════════════════════════════════
#  🔒 Segurança
# ═══════════════════════════════════════════════════════════════
@pytest.mark.seguranca
def test_nenhuma_resposta_expoe_campo_interno(cliente, catalogo):
    """🔒 Varre TODAS as rotas de leitura de uma vez."""
    proibidos = {"custo", "senha", "senha_hash", "fornecedor"}
    corpos = [cliente.get("/produtos").json(),
              cliente.get("/produtos/NB-DELL-15").json()]
    for corpo in corpos:
        itens = corpo if isinstance(corpo, list) else [corpo]
        for item in itens:
            vazou = proibidos & set(item)
            assert not vazou, f"vazou {vazou} em {item}"


@pytest.mark.seguranca
@pytest.mark.parametrize("limite", [0, -1, 101, 999999])
def test_limite_fora_da_faixa_e_recusado(cliente, limite):
    assert cliente.get(f"/produtos?limite={limite}").status_code == 422


@pytest.mark.seguranca
@pytest.mark.parametrize("sku", [
    "nb-dell-15",                 # minúsculo
    "ab",                         # curto demais
    "NB DELL 15",                 # espaço
    "NB-DELL-15'; DROP TABLE produtos--",
])
def test_sku_fora_do_padrao_e_recusado(cliente, sku):
    resposta = cliente.post("/produtos", json={
        "sku": sku, "nome": "Teste", "preco": 100.0, "custo": 50.0})
    assert resposta.status_code == 422


# ═══════════════════════════════════════════════════════════════
#  Regras de negócio
# ═══════════════════════════════════════════════════════════════
def test_preco_abaixo_do_custo_e_recusado(cliente):
    resposta = cliente.post("/produtos", json={
        "sku": "XX-TESTE-1", "nome": "Prejuízo", "preco": 50.0, "custo": 100.0})
    assert resposta.status_code == 422


def test_sku_duplicado_e_conflito(cliente, catalogo):
    resposta = cliente.post("/produtos", json={
        "sku": "NB-DELL-15", "nome": "Cópia", "preco": 3000.0, "custo": 2000.0})
    assert resposta.status_code == 409


@pytest.mark.parametrize("peso,cep,esperado", [
    (2.0, "13010-000", {"prazo_dias": 2, "gratis": False}),
    (1.0, "88010-000", {"prazo_dias": 6, "gratis": False}),
])
def test_cotacao_pela_api(cliente, peso, cep, esperado):
    corpo = cliente.post("/cotacoes", json={"peso_kg": peso, "cep": cep}).json()
    for chave, valor in esperado.items():
        assert corpo[chave] == valor


def test_cotacao_invalida_da_422(cliente):
    assert cliente.post("/cotacoes",
                        json={"peso_kg": 1.0, "cep": "abc"}).status_code == 422


# ═══════════════════════════════════════════════════════════════
#  Integração externa
# ═══════════════════════════════════════════════════════════════
@respx.mock
def test_transportadora_indisponivel_nao_derruba_nada():
    """🎯 A falha do parceiro vira uma exceção NOSSA, tratável."""
    from app.transportadora import TransportadoraIndisponivel, cotar_frete

    respx.post("https://api.veloz.com.br/v1/cotacoes").mock(
        side_effect=httpx.ConnectError("fora do ar"))

    with pytest.raises(TransportadoraIndisponivel):
        cotar_frete(1.0, "13010-000")


# ═══════════════════════════════════════════════════════════════
#  Regressão
# ═══════════════════════════════════════════════════════════════
def test_regressao_142_frete_gratis_mantem_prazo():
    """Bug #142: frete grátis zerava o prazo junto com o valor.

    💭 Este teste nasceu de um chamado real. Ele existe para que o bug
       não volte na próxima refatoração — e o número no nome liga o
       teste ao histórico.
    """
    from app.frete import calcular_frete

    resultado = calcular_frete(2.0, "88010-000", valor_pedido=800.0)
    assert resultado["valor"] == 0.0
    assert resultado["prazo_dias"] == 6          # 🔴 o que quebrava

In [ ]:
pytest_("-m", "not lento", "-v", resumo=False, linhas=80)

In [ ]:
print("Estrutura final do projeto:\n")
arvore(BASE)

In [ ]:
# O comando que roda no CI (Módulo 09)
print("═══ como o CI vai rodar ═══\n")
codigo = pytest_("-q", "--tb=short", "-m", "not integracao", resumo=False)
print(f"\n{'✅ CI verde' if codigo == 0 else '🔴 CI vermelho'} "
      f"(código de saída {codigo})")
print("\n💡 O código de saída é tudo que o CI olha:")
print("   0 = pode publicar   ·   qualquer outro = pare.")

## 📝 Exercícios

**E1.** Escreva 5 testes para `calcular_frete` cobrindo bordas que a aula não cobriu (peso exatamente 30, CEP com ponto, valor negativo).

**E2.** Converta 5 testes repetitivos em um `parametrize` com `id` legível para cada caso.

**E3.** Crie uma fixture `produto_com_estoque_zerado` e use-a em três testes diferentes.

**E4.** 🔴 Prove que o isolamento funciona: crie 3 produtos num teste e verifique, em outro, que o banco está vazio. Depois remova o `rollback` da fixture e mostre a quebra.

**E5.** Escreva uma fixture com `scope="module"` que devolva um objeto mutável e demonstre a contaminação. Depois corrija.

**E6.** Use `pytest.raises` com `match=` para verificar as 4 mensagens de erro do `FreteInvalido`.

**E7.** 🔴 Teste a autorização da API do M06: para cada rota de escrita, verifique anônimo=401, leitor=403, operador=200.

**E8.** Escreva um teste que leia o `openapi.json` e falhe se qualquer esquema de **resposta** contiver `custo`.

**E9.** Com `respx`, teste que o seu cliente **não** repete um `404`.

**E10.** Com `respx`, verifique os cabeçalhos enviados (`User-Agent`, `Authorization`) em cada chamada.

**E11.** 🔴 Teste a atomicidade do `POST /pedidos`: item viável + item sem estoque → nada muda.

**E12.** Teste a validação de webhook (07_02): assinatura inválida, timestamp velho e evento repetido.

**E13.** Escreva um teste de upload que verifique a rejeição de magic bytes de executável.

**E14.** Rode `--cov` e encontre a linha menos coberta do seu projeto. Escreva o teste que falta.

**E15.** Escreva um teste que passe por 100% das linhas sem verificar nada, e explique por que a cobertura é enganosa.

**E16.** Configure `--strict-markers` e prove que um marcador com erro de digitação vira erro.

In [ ]:
# E1

In [ ]:
# E2

In [ ]:
# E3

In [ ]:
# E4

In [ ]:
# E5

In [ ]:
# E6

In [ ]:
# E7

In [ ]:
# E8

In [ ]:
# E9

In [ ]:
# E10

In [ ]:
# E11

In [ ]:
# E12

In [ ]:
# E13

In [ ]:
# E14

In [ ]:
# E15

In [ ]:
# E16

## 📋 Cola de referência

```python
# ═══ Estrutura ═══
# projeto/
# ├── app/
# ├── tests/
# │   ├── conftest.py      ← fixtures compartilhadas (o pytest acha sozinho)
# │   └── test_*.py
# └── pyproject.toml       ← [tool.pytest.ini_options]

# ═══ Comandos ═══
# pytest                       tudo
# pytest -v                    com o nome de cada teste
# pytest -x                    para na primeira falha
# pytest -k "frete and not lento"
# pytest -m seguranca          por marcador
# pytest --lf                  só os que falharam da última vez
# pytest --cov=app --cov-report=term-missing
# pytest -q --tb=short         formato de CI

# ═══ Fixtures ═══
@pytest.fixture               # scope="function" (padrão) → isolamento
@pytest.fixture(scope="session")   # 🔶 só para caro E imutável
def recurso():
    preparar()
    yield objeto
    limpar()                  # roda mesmo se o teste falhar

# do próprio pytest: tmp_path · monkeypatch · capsys · caplog

# ═══ 🎯 A fixture que importa: banco isolado ═══
# SQLite em memória → recrie o banco por teste (simples e imbatível)
@pytest.fixture
def sessao():
    m = create_engine("sqlite:///:memory:",
                      connect_args={"check_same_thread": False},
                      poolclass=StaticPool)      # 🔴 obrigatório
    Base.metadata.create_all(m)
    s = sessionmaker(bind=m, expire_on_commit=False)()
    yield s
    s.close(); m.dispose()

# PostgreSQL → transação + rollback (recriar seria lento demais)
@pytest.fixture
def sessao(motor):                                # motor: scope="session"
    conexao = motor.connect(); trans = conexao.begin()
    s = sessionmaker(bind=conexao, expire_on_commit=False,
                     join_transaction_mode="create_savepoint")()
    yield s
    s.close(); trans.rollback(); conexao.close()   # 🔑 desfaz tudo
# 🔴 No SQLite este 2º padrão exige desligar o controle de transação do
#    pysqlite:  event "connect" → dbapi.isolation_level = None
#               event "begin"   → conexao.exec_driver_sql("BEGIN")

@pytest.fixture
def cliente(sessao):
    app = criar_app()
    app.dependency_overrides[obter_sessao] = lambda: sessao
    with TestClient(app) as c: yield c
    app.dependency_overrides.clear()

# ═══ parametrize ═══
@pytest.mark.parametrize("entrada,esperado", [(1, 2), (2, 4)])
@pytest.mark.parametrize(..., [pytest.param(500, True, id="no_limite")])

# ═══ Erros ═══
with pytest.raises(FreteInvalido, match="CEP inválido"):
    ...
pytest.approx(17.60)          # 🎯 float não compara com ==

# ═══ respx: HTTP sem rede ═══
@respx.mock
def test_x():
    rota = respx.post(URL).mock(return_value=httpx.Response(200, json={...}))
    rota = respx.post(URL).mock(side_effect=[Response(503), Response(200)])
    rota = respx.post(URL).mock(side_effect=httpx.ConnectError("x"))
    assert rota.call_count == 1               # 🎯 testa o que NÃO fez
    rota.calls[0].request.content

# ═══ Marcadores ═══
@pytest.mark.lento / .integracao / .seguranca
@pytest.mark.skip(reason=...) / .skipif(cond, reason=...)
@pytest.mark.xfail(reason="bug #142")
# addopts = "--strict-markers"   🔴 senão um typo vira silêncio

# ═══ Cobertura 🔴 ═══
# baixa → há buraco (verdade útil)
# alta  → NÃO significa testado
```

## ✅ Checklist de saída

- [ ] Sei que testes existem para dar **coragem de mudar**
- [ ] Escrevo no padrão Arrange–Act–Assert
- [ ] Meus nomes de teste descrevem a regra
- [ ] Uso fixtures em vez de repetir preparo
- [ ] Sei onde mora o `conftest.py` e por quê
- [ ] 🔴 **Escopo amplo só para o que é caro E imutável**
- [ ] 🎯 **Uso `dependency_overrides` para trocar o banco**
- [ ] 🔴 **Lembro do `poolclass=StaticPool` com SQLite em memória**
- [ ] Cada teste começa com o banco no mesmo estado
- [ ] Tenho um teste que **prova** o isolamento
- [ ] Uso `parametrize` com `id` legível
- [ ] Testo as **bordas** dos limites
- [ ] Uso `pytest.approx` para decimais
- [ ] Uso `pytest.raises(..., match=...)`
- [ ] Testo os caminhos de **erro**, não só o feliz
- [ ] 🔒 **Tenho um teste que prova que campo interno não vaza**
- [ ] Uso `respx` em vez de depender da internet
- [ ] Verifico o que foi **enviado**, não só o que voltou
- [ ] Testo o que o código **não** faz (`call_count == 1`)
- [ ] Uso marcadores e `--strict-markers`
- [ ] Não testo bibliotecas nem detalhes internos
- [ ] 🔴 **Sei que cobertura alta não significa código testado**
- [ ] Escrevo um teste que falha **antes** de corrigir um bug

---

### ➡️ Próxima aula

**`07_99_Lista_Exercicios.ipynb`** — A lista do módulo e a evolução do Atlas: integrações resilientes, webhooks, cache e a suíte de testes que dá coragem para o Módulo 08.